# MLP (Pipeline B)

**Model**: `MLPRegressor` / **Target**: gdpc1  
**Variables**: Cat3 (53 + COVID = 56) / **n_lags**: 4  
**Scaling**: YES (gradient conditioning) / **HP tuning**: YES  
**10-model averaging**: YES  
**Note**: 212 features for ~194 obs. Alpha grid includes 1e-2 for stronger L2 reg.

In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import norm
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]
sys.path.insert(0, os.path.join(os.path.pardir, 'data'))
from helpers import (gen_lagged_data, gen_vintage_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
    get_features, split_for_scaler, load_data, PREDICTIONS_DIR, TARGET, COVID)
SEED = 42; np.random.seed(SEED)
TRAIN_START = '1959-01-01'; TRAIN_END = '2007-12-31'; VAL_END = '2016-12-31'
TEST_START = '2017-01-01'; TEST_END = '2025-12-31'
N_LAGS = 4; N_MODELS = 20; VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1}
print('MLP — Cat3, n_lags=4, scaling=YES')

MLP — Cat3, n_lags=4, scaling=YES


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
scaled_cols, unscaled_cols = split_for_scaler(features)
# Phase A: HP tune alpha on validation
tune = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_f = mean_fill_dataset(tune, tune)
tune_fl = flatten_data(tune_f, TARGET, N_LAGS)
tune_fl = tune_fl.loc[tune_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)
fcols = [c for c in tune_fl.columns if c != 'date' and c != TARGET]
sc_c = [c for c in fcols if c in scaled_cols or any(c.startswith(s + '_') for s in scaled_cols)]
us_c = [c for c in fcols if c in unscaled_cols or any(c.startswith(u + '_') for u in unscaled_cols)]
tscv = TimeSeriesSplit(n_splits=5)
best_score = np.inf; best_alpha = 0.001; best_hls = (64,)
for hls in [(16,), (32,), (32, 16)]:
    for alpha in [1e-2, 1e-1, 1.0]:
        scores = []
        for tr, va in tscv.split(tune_fl):
            sc = RobustScaler(quantile_range=(10, 90))
            X_tr_s = sc.fit_transform(tune_fl[sc_c].values[tr])
            X_tr = np.hstack([X_tr_s, tune_fl[us_c].values[tr]]) if us_c else X_tr_s
            X_va_s = sc.transform(tune_fl[sc_c].values[va])
            X_va = np.hstack([X_va_s, tune_fl[us_c].values[va]]) if us_c else X_va_s
            m = MLPRegressor(hidden_layer_sizes=hls, alpha=alpha, random_state=SEED, max_iter=1500, early_stopping=True, validation_fraction=0.2, n_iter_no_change=30, learning_rate_init=0.0005)
            m.fit(X_tr, tune_fl[TARGET].values[tr])
            p = m.predict(X_va)
            scores.append(np.mean((p - tune_fl[TARGET].values[va])**2))
        if np.mean(scores) < best_score:
            best_score = np.mean(scores)
            best_alpha = alpha
            best_hls = hls
print(f'Best MLP params: alpha={best_alpha}, hidden_layer_sizes={best_hls}')

C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


...

C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


Best MLP params: alpha=1.0, hidden_layer_sizes=(32, 16)


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [3]:
test_quarters = monthly[(monthly['date'] >= TEST_START) & (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3,6,9,12])]['date'].tolist()
predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0
for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date'] == pd_q][TARGET].values[0])
    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train_fl, tr_fl, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, fc, N_LAGS
        )
        if len(train_fl) < 10:
            predictions[vn].append(np.nan); continue
        try:
            sc = RobustScaler(quantile_range=(10, 90))
            X_tr_s = sc.fit_transform(train_fl[sc_c].values)
            X_tr = np.hstack([X_tr_s, train_fl[us_c].values]) if us_c else X_tr_s
            models = []
            for k in range(N_MODELS):
                m = MLPRegressor(hidden_layer_sizes=best_hls, activation='relu', solver='adam',
                    alpha=best_alpha, learning_rate_init=0.0005, learning_rate='adaptive',
                    early_stopping=True, validation_fraction=0.2, n_iter_no_change=30,
                    random_state=SEED+k, max_iter=1500)
                m.fit(X_tr, train_fl[TARGET].values); models.append(m)
            X_te_s = sc.transform(tr_fl[sc_c].values)
            X_te = np.hstack([X_te_s, tr_fl[us_c].values]) if us_c else X_te_s
            preds = [m.predict(X_te)[0] for m in models]
            predictions[vn].append(np.nanmedian(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1
    if (i+1) % 6 == 0:
        print(f'  {i+1}/{len(test_quarters)}')
print(f'Done. {failed} failures.')

C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


...

C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


  24/36


  30/36


  36/36
Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR, f'mlp_{vn}.csv'), index=False)
    print(f'Saved mlp_{vn}.csv')

Saved mlp_m1.csv
Saved mlp_m2.csv
Saved mlp_m3.csv
Saved mlp_post1.csv


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum() > 0 else np.nan
def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum() > 0 else np.nan

panels = {
    'Pre-COVID  (2017-2019)': ('2017-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2017-2025)': ('2017-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print(f'MLP best params: alpha={best_alpha}, hls={best_hls}')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print(f'--- {pn} ---')
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print(f'  {vn}  RMSFE={rmse(a[m], pp[m]):.6f}  MAE={mae(a[m], pp[m]):.6f}')

MLP best params: alpha=1.0, hls=(32, 16)
--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.002910  MAE=0.002354
  m2  RMSFE=0.003141  MAE=0.002514
  m3  RMSFE=0.003170  MAE=0.002507
  post1  RMSFE=0.003162  MAE=0.002526
--- COVID      (2020-2021) ---
  m1  RMSFE=0.042787  MAE=0.026668
  m2  RMSFE=0.032545  MAE=0.020570
  m3  RMSFE=0.028836  MAE=0.017708
  post1  RMSFE=0.027241  MAE=0.017910
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.004835  MAE=0.003442
  m2  RMSFE=0.004938  MAE=0.003838
  m3  RMSFE=0.004804  MAE=0.003697
  post1  RMSFE=0.005043  MAE=0.003776
--- Full test  (2017-2025) ---
  m1  RMSFE=0.019523  MAE=0.008076
  m2  RMSFE=0.015187  MAE=0.007085
  m3  RMSFE=0.013650  MAE=0.006475
  post1  RMSFE=0.012926  MAE=0.006478
